# AIS data exploration and preprocessing

Last week, we plotted reported vessel positions on a map. This gave us a first impression of the data and highlighted locations that might need closer inspection. A map alone, however, does not tell us how regularly positions were reported or whether a large jump occurred over seconds or several minutes.

This week, we use statistical analysis to investigate these questions. We explore the distributions of time intervals and spatial distances between consecutive reports, then use what we learn to guide the removal of repeated records and possible drift points. The aim is to understand the evidence behind each cleaning decision.

Sections 1–3 revisit loading, inspection and visualization. Sections 4–7 develop the temporal and spatial analysis and preprocessing.

Keep this notebook alongside the extracted project's `README.md` and `libs` folder. Update the dataset path below to match your files.


# 1. Import the necessary libraries

<div style="background-color:#fff3e0; border-left:5px solid #e6a04b; padding:12px 16px; border-radius:4px;">

<b>How the code works</b>
<p>As last week, <code>pandas</code> handles the tabular data, <code>matplotlib.pyplot</code> displays the plots, and <code>plot_on_map</code> creates the background map.</p>
<p><b>New this week:</b> the imports from <code>time_processing</code>, <code>distance_calculation</code> and <code>drift_detection</code> provide the functions for temporal analysis, spatial analysis and cleaning. All imports are collected here; each new function's parameters and returned data are explained where it is used.</p>
<p>If Cartopy is not installed, uncomment <code># %pip install cartopy</code>, run it, then restart the kernel and rerun the notebook.</p>
<p>If you get a <code>PROJ</code> error when importing <code>plot_on_map</code>, uncomment the lines starting with <code># import os, pyproj</code> below. Then restart the kernel and rerun the notebook from the beginning.</p>
</div>


In [ ]:
# # %pip install cartopy

# # Uncomment these lines if you get a PROJ error when importing plot_on_map:
# # import os, pyproj
# # proj_dir = pyproj.datadir.get_data_dir()
# # os.environ["PROJ_LIB"] = proj_dir
# # os.environ["PROJ_DATA"] = proj_dir

# import pandas as pd
# import matplotlib.pyplot as plt

# # Map visualization from last week.
# from libs.plot_traffic_on_map import plot_on_map

# # Functions introduced this week.
# from libs.time_processing import (convert_timeStr_pdtime, sort_data_vel_time,
#                                   calculate_time_difference, remove_time_repeat_data)
# from libs.distance_calculation import calculate_spatial_distance
# from libs.drift_detection import detect_trajectory_drift_point


# 2. Load and inspect the dataset

Before changing anything, inspect the available columns, sample records and data types. Consider whether the data are ready for calculations involving time and position.


In [ ]:
# # Put the right file path here if the CSV is stored elsewhere.
# readFromFile = "AIS_data_timestamp.csv"
# df_ship_AIS_data = pd.read_csv(readFromFile)

# # View the first five records.
# df_ship_AIS_data.head(5)

In [ ]:
# # Inspect the columns and their data types.
# df_ship_AIS_data.info()

<div style="background-color:#e6f0ff; padding:10px; border:1px solid #99ccff;">
❓ <b>Discussion Questions:</b>
<ol>
<li>Which column has a data type that needs attention before we can calculate time intervals?</li>
<li>Are any values missing in the columns shown? Does the absence of missing values mean that all records are reliable?</li>
</ol>
💡 <i>Write your answers in the cell below.</i>
</div>

Answers:


# 3. Visualize the reported positions

Revisit last week's map to identify positions that deserve further investigation. At this stage, we are looking for possible issues, without deciding which records to remove.

Create the background map, add the reported longitude and latitude values as a scatter plot, and display the result. A scatter plot shows the observations without assuming a path between them.


<div style="background-color:#fff3e0; border-left:5px solid #e6a04b; padding:12px 16px; border-radius:4px;">

<b>How the code works</b>
<ul>
<li><code>plot_on_map()</code> creates a map using its default boundaries and returns <code>fig</code> (the figure) and <code>ax</code> (the map axes). An optional <code>extent</code> argument sets boundaries in the order [minimum longitude, maximum longitude, minimum latitude, maximum latitude].</li>
<li><code>df_ship_AIS_data.plot()</code> adds the AIS positions to the map using the following parameters:
<ul>
<li><code>ax=ax</code>: plots on the existing map axes.</li>
<li><code>x="lon"</code>: uses the longitude column for the horizontal coordinates.</li>
<li><code>y="lat"</code>: uses the latitude column for the vertical coordinates.</li>
<li><code>kind="scatter"</code>: shows individual positions without connecting them with lines.</li>
<li><code>s=5</code>: sets the marker size.</li>
<li><code>zorder=3</code>: places the points above the background map layers.</li>
</ul>
</li>
<li><code>ax.set_title()</code> adds a title, and <code>plt.show()</code> displays the completed figure.</li>
</ul>
</div>

In [ ]:
# # 1. Use the function described in README.md to create the background map.
# fig, ax = plot_on_map()

# # 2. Plot the AIS longitude and latitude values on the map axes.
# df_ship_AIS_data.plot(kind="scatter", ax=ax, x="lon", y="lat", s=5, zorder=3)
# ax.set_title("Reported AIS positions")



# # 3. Display the map with the AIS points added.
# plt.show()

<div style="background-color:#e6f0ff; padding:10px; border:1px solid #99ccff;">
❓ <b>Discussion Questions:</b>
<ol>
<li>What do you notice about the positions relative to the coastline and the main cluster of observations? Are there locations you would investigate further?</li>
<li>Can this map alone tell us whether an isolated position is an error or a genuine visit to another location? What information is missing?</li>
</ol>
💡 <i>Write your answers in the cell below.</i>
</div>

Answers:


# 4. Explore the temporal characteristics

To interpret the positions on the map, we first need to understand when they were recorded. The time between reports tells us how regularly the vessel was observed and whether there are gaps in the record.

The timestamp column is currently stored as an object. We convert it to datetime values and add UNIX epoch time so that elapsed time can be calculated. We then sort the records by MMSI and time, ensuring that consecutive observations belong to the same vessel and follow chronological order.


<div style="background-color:#fff3e0; border-left:5px solid #e6a04b; padding:12px 16px; border-radius:4px;">

<b>How the code works</b>
<p><code>convert_timeStr_pdtime(df_ais, time_column_name, time_format)</code> prepares the timestamps.</p>
<ul>
<li><code>df_ais</code>: the input pandas DataFrame. We pass a copy of the original data because the function modifies its input.</li>
<li><code>time_column_name="timestamp"</code>: the column containing the original timestamps.</li>
<li><code>time_format="%Y-%m-%d %H:%M:%S"</code>: the year-month-day and hour-minute-second format used in this CSV.</li>
<li><b>Returns:</b> a pandas DataFrame with <code>timestamp</code> converted to datetime and a new integer column, <code>ts</code>, containing UNIX epoch time in seconds.</li>
</ul>
<p><code>sort_data_vel_time(df_ais, vessel_column_name, time_column_name)</code> orders the records.</p>
<ul>
<li><code>df_ais</code>: the DataFrame with converted timestamps.</li>
<li><code>vessel_column_name="mmsi"</code>: the vessel identifier used as the first sorting key.</li>
<li><code>time_column_name="ts"</code>: the time column used to sort records within each vessel.</li>
<li><b>Returns:</b> a pandas DataFrame sorted in ascending order by vessel and time, with the same columns.</li>
</ul>
</div>

In [ ]:
# # Preserve the original data for the final comparison.
# df_ais = convert_timeStr_pdtime(
#     df_ship_AIS_data.copy(), "timestamp", "%Y-%m-%d %H:%M:%S"
# )
# df_ais = sort_data_vel_time(df_ais, "mmsi", "ts")
# df_ais.info()


With the records in order, we can calculate the time between consecutive reports. Rather than inspecting individual intervals only, we summarize their distribution and plot a histogram. This helps us distinguish typical reporting intervals from zero intervals and unusually long gaps. No records have been removed yet.


<div style="background-color:#fff3e0; border-left:5px solid #e6a04b; padding:12px 16px; border-radius:4px;">

<b>How the code works</b>
<p><code>calculate_time_difference(df_ais, vessel_column_name, time_column_name, time_grain)</code> calculates elapsed time within each vessel.</p>
<ul>
<li><code>df_ais</code>: the sorted pandas DataFrame.</li>
<li><code>vessel_column_name="mmsi"</code>: groups records by vessel before taking differences.</li>
<li><code>time_column_name="ts"</code>: the UNIX timestamp column, measured in seconds.</li>
<li><code>time_grain="m"</code>: divides differences by 60 to express them in minutes.</li>
<li><b>Returns:</b> the pandas DataFrame with a new numeric column, <code>time_difference(min)</code>. The function modifies the input DataFrame.</li>
</ul>
<p>The first record of each vessel has no preceding record, so its interval is <code>NaN</code>. <code>.describe()</code> returns summary statistics as a pandas Series. <code>.hist(bins=50, edgecolor="black")</code> plots frequencies in 50 bins and returns a Matplotlib Axes. Both omit missing intervals.</p>
</div>

In [ ]:
# df_ais = calculate_time_difference(df_ais, "mmsi", "ts", "m")
# print(df_ais["time_difference(min)"].describe())
# print("Zero time intervals:", df_ais["time_difference(min)"].eq(0).sum())

# df_ais["time_difference(min)"].hist(bins=50, edgecolor="black")
# plt.xlabel("Time difference (minutes)")
# plt.ylabel("Frequency")
# plt.title("Distribution of reporting intervals before cleaning")
# plt.show()


<div style="background-color:#e6f0ff; padding:10px; border:1px solid #99ccff;">
❓ <b>Discussion Questions:</b>
<ol>
<li>What do the summary statistics and histogram tell you about typical reporting intervals and their variability?</li>
<li>What could zero intervals and unusually long gaps represent? Should both lead to the same cleaning decision?</li>
</ol>
💡 <i>Write your answers in the cell below.</i>
</div>

Answers:


# 5. Handle repeated timestamps

The temporal analysis helps us identify records that share a timestamp. Before removing them, we should check what they contain. Identical records repeat the same observation, whereas different positions at the same timestamp create a conflict that requires a decision.

Inspect the repeated vessel–timestamp pairs below. 


<div style="background-color:#fff3e0; border-left:5px solid #e6a04b; padding:12px 16px; border-radius:4px;">

<b>How the code works</b>
<p><code>.duplicated(subset=["mmsi", "timestamp"], keep=False)</code> returns a Boolean pandas Series identifying all records in repeated vessel–timestamp pairs. <code>.loc[repeated_timestamp, ...]</code> selects these rows and the columns to inspect, returning a DataFrame.</p>

</div>

In [ ]:
# # Inspect all records involved in repeated vessel–timestamp pairs.
# repeated_timestamp = df_ais.duplicated(subset=["mmsi", "timestamp"], keep=False)
# df_ais.loc[repeated_timestamp, ["mmsi", "timestamp", "lon", "lat"]]


<div style="background-color:#e6f0ff; padding:10px; border:1px solid #99ccff;">
❓ <b>Discussion Questions:</b>
<ol>
<li>Do the repeated timestamps contain identical positions or conflicting positions? What does this imply for retaining the first record?</li>
<li>Why should a long reporting gap be investigated separately from repeated timestamps?</li>
</ol>
💡 <i>Write your answers in the cell below.</i>
</div>

Answers:


For this exercise, we retain the first record for each matching pair and remove the subsequent records. This is a simple cleaning rule; consider whether the inspected records support it.

<div style="background-color:#fff3e0; border-left:5px solid #e6a04b; padding:12px 16px; border-radius:4px;">

<b>How the code works</b>

<p><code>remove_time_repeat_data(df_ais, time_diff_column_name)</code> applies the cleaning rule.</p>
<ul>
<li><code>df_ais</code>: the chronologically sorted DataFrame with time differences calculated within each vessel.</li>
<li><code>time_diff_column_name="time_difference(min)"</code>: the column containing elapsed time, and used to identify zero intervals.</li>
<li><b>Returns:</b> a filtered pandas DataFrame excluding zero-interval rows. The first row for each vessel, whose interval is <code>NaN</code>, is retained. No new columns are added.</li>
</ul>
<p><code>.copy()</code> creates an independent copy of the filtered data. <code>len()</code> returns the number of rows, allowing us to report how many records were removed.</p>
</div>

In [ ]:
# rows_before = len(df_ais)
# df_ais = remove_time_repeat_data(df_ais, "time_difference(min)").copy()
# print("Records removed with repeated timestamps:", rows_before - len(df_ais))


# 6. Explore the spatial characteristics

We now have an ordered sequence without repeated timestamps. Returning to the unusual positions on the map, we can investigate how far apart consecutive observations are. Are most reported movements small, with a few much larger jumps?

Calculate the distances between consecutive positions, then examine their summary statistics and histogram. Here, spatial analysis means studying the distribution of these distances. A large distance is a reason to investigate, but its interpretation also depends on how much time passed between reports. We have not removed any drift points at this stage.


<div style="background-color:#fff3e0; border-left:5px solid #e6a04b; padding:12px 16px; border-radius:4px;">

<b>How the code works</b>
<p><code>calculate_spatial_distance(df_ais, vessel_column_name)</code> calculates distances within each vessel using the haversine formula.</p>
<ul>
<li><code>df_ais</code>: the sorted pandas DataFrame containing longitude and latitude in degrees in columns named <code>lon</code> and <code>lat</code>.</li>
<li><code>vessel_column_name="mmsi"</code>: identifies the vessel groups, preventing comparisons across vessels.</li>
<li><b>Returns:</b> a pandas DataFrame with a new numeric column, <code>spatial_distance(km)</code>, giving the great-circle distance from the preceding position. The first distance per vessel is <code>NaN</code>.</li>
</ul>
<p><code>.reset_index(drop=True)</code> restores a simple row index after the grouped calculation. <code>.describe()</code> returns summary statistics; <code>.hist(bins=10, ...)</code> returns a Matplotlib Axes showing the distance frequencies in 10 bins. Distances connect recorded positions and do not capture turns between reports.</p>
</div>

In [ ]:
# df_ais = calculate_spatial_distance(df_ais, "mmsi").reset_index(drop=True)
# print(df_ais["spatial_distance(km)"].describe())

# df_ais["spatial_distance(km)"].hist(bins=10, edgecolor="black", figsize=(6, 6))
# plt.xlabel("Spatial distance (km)")
# plt.ylabel("Frequency")
# plt.title("Distribution of spatial distances before drift removal")
# plt.grid(True)
# plt.show()


The histogram helps us identify distances that stand apart from most observations. To investigate those distances, look at the corresponding records and reporting intervals together. This connects the temporal and spatial analyses before we make a filtering decision.


<div style="background-color:#fff3e0; border-left:5px solid #e6a04b; padding:12px 16px; border-radius:4px;">

<b>How the code works</b>
<p><code>.nlargest(10, "spatial_distance(km)")</code> returns a pandas DataFrame containing the ten rows with the largest distances, ordered from largest to smallest. Selecting the columns below lets us compare each distance with its elapsed time and reported position.</p>
</div>

In [ ]:
# df_ais.nlargest(10, "spatial_distance(km)")[
#     ["mmsi", "timestamp", "lon", "lat", "time_difference(min)", "spatial_distance(km)"]
# ]


<div style="background-color:#e6f0ff; padding:10px; border:1px solid #99ccff;">
❓ <b>Discussion Questions:</b>
<ol>
<li>What distinguishes the largest distances from the rest? What distance threshold would you investigate, and what evidence would you need to justify it?</li>
<li>Do the elapsed times help explain the largest distances, or make them more suspicious?</li>
<li>Would applying an IQR rule separately to latitude and longitude identify the same issues? Could it flag a legitimate location?</li>
</ol>
💡 <i>Write your answers in the cell below.</i>
</div>

Answers:


# 7. Investigate and remove drift points

The distributions have helped us identify movements that deserve closer inspection. We now move from exploration to a cleaning rule. An isolated erroneous position can produce two large distances: a jump away from the surrounding positions and a jump back towards them.

The supplied detector looks for this pattern. Its thresholds are already defined, so compare them with your interpretation of the spatial distribution rather than assuming they are suitable for every dataset. Inspect the flagged records and their incoming and outgoing distances before filtering them.


<div style="background-color:#fff3e0; border-left:5px solid #e6a04b; padding:12px 16px; border-radius:4px;">

<b>How the code works</b>
<p><code>detect_trajectory_drift_point(df_ais, vessel_column_name)</code> flags possible isolated drift points.</p>
<ul>
<li><code>df_ais</code>: the sorted pandas DataFrame containing <code>spatial_distance(km)</code>.</li>
<li><code>vessel_column_name="mmsi"</code>: identifies the vessel groups used to calculate statistics and examine neighbouring distances.</li>
<li><b>Returns:</b> a pandas DataFrame with <code>spatial_distance_next(km)</code>, the distance to the following position, and <code>drift_point</code>, an integer flag: <code>1</code> for a candidate and <code>0</code> otherwise.</li>
</ul>
<p>The rule requires <b>both distances to be at least 15 km</b> and <b>both absolute z-scores to exceed 3</b>. The mean and standard deviation come from the next-distance column within each vessel. These thresholds are fixed inside the supplied function, not passed as parameters or estimated from our histogram. The rule does not consider elapsed time.</p>
<p><code>.loc[df_ais["drift_point"] == 1]</code> returns the flagged rows as a DataFrame for inspection. A flag is evidence to investigate, not proof of an error.</p>
</div>

In [ ]:
# df_ais = detect_trajectory_drift_point(df_ais, "mmsi")
# df_drift_points = df_ais.loc[df_ais["drift_point"] == 1].copy()
# print("Flagged drift points:", len(df_drift_points))
# df_drift_points[["mmsi", "timestamp", "lon", "lat", "time_difference(min)",
#                  "spatial_distance(km)", "spatial_distance_next(km)", "drift_point"]]


<div style="background-color:#e6f0ff; padding:10px; border:1px solid #99ccff;">
❓ <b>Discussion Questions:</b>
<ol>
<li>Do the flagged distances and reporting intervals support removing these observations? How does the supplied threshold compare with the distribution you examined?</li>
<li>Why does the rule examine both incoming and outgoing distances? Could it miss smaller errors or several consecutive erroneous positions?</li>
</ol>
💡 <i>Write your answers in the cell below.</i>
</div>

Answers:


For this exercise, apply the rule by removing the flagged points and keeping the remaining observations in a new dataset. 

<div style="background-color:#fff3e0; border-left:5px solid #e6a04b; padding:12px 16px; border-radius:4px;">

<b>How the code works</b>
<p><code>.loc[df_ais["drift_point"] == 0].copy()</code> returns an independent pandas DataFrame of retained records. <code>.reset_index(drop=True)</code> assigns a new sequential row index.</p>
</div>

In [ ]:
# df_ship_AIS_data_afterpre = df_ais.loc[df_ais["drift_point"] == 0].copy()
# df_ship_AIS_data_afterpre = df_ship_AIS_data_afterpre.reset_index(drop=True)
# print("Original records:", len(df_ship_AIS_data))
# print("Records after preprocessing:", len(df_ship_AIS_data_afterpre))

<div style="background-color:#e8f5e9; border-left:5px solid #43a047; padding:14px 18px; border-radius:6px;">
<b>✏️ Your turn: Return to the map</b>
<p>Write your own code to visualize the filtered positions, using the same boundaries as the original map. Refer to Section 3 if needed.</p>
<p>Compare the filtered positions with the original map. Are the changes supported by the statistical evidence from your temporal and spatial analyses?</p>
<i>Write your code in the cell below.</i>
</div>

In [ ]:
# # Write your code here


<div style="background-color:#e6f0ff; padding:10px; border:1px solid #99ccff;">
❓ <b>Discussion Questions:</b>
<ol>
<li>How has the map changed? Explain how the temporal and spatial analyses support your cleaning decisions.</li>
<li>Does a more plausible-looking map mean that all errors have been removed? Which assumptions and limitations would you report?</li>
</ol>
💡 <i>Write your answers in the cell below.</i>
</div>

Answers:


<div style="background-color:#fff3cd; border-left:6px solid #f0ad4e; padding:14px 18px; border-radius:4px;">

<h2 style="margin-top:0;">⚠️ Applying these examples</h2>

<p>
The examples demonstrate how selected EDA and cleaning methods can be implemented. 
For the assignment, use your own EDA findings and the purpose of the analysis 
to decide which issues require action and which treatment is appropriate.
</p>

<p style="margin-bottom:0;">
<strong>Do not apply a method only because it was demonstrated here.</strong>
</p>

</div>